---
title: "Regression"
subtitle: "From Excel to Python"
execute:
  echo: true
---

**In this lesson**

- "Falling" is a slope: turning the board's claim into a line
- A table of days, and a correlation matrix before any model
- One line through trips per day: slope, intercept, R-squared, and the Excel trendline
- Reading a regression table: the coefficient, its standard error, its interval
- Residuals: what the line gets wrong, and the pattern in it
- Adding weekday: categorical variables, dummies, and what each coefficient means
- What is left over: holidays, weather, and the limit of the columns you have
- Extrapolation, and why the line stops at the edge of the data
- Excel's `SLOPE`, `LINEST`, and the trendline, and where they stop

**The one idea**

> A regression is a sentence of the form "each extra unit of this comes with that many units of the other, holding the rest fixed", with a standard error attached. The coefficient is the finding; R-squared says how much of the day-to-day story your columns can tell; the residuals say what they cannot. Read all three before believing any of them.

::: {.content-visible when-format="html"}
[Download this notebook](chapter_08_lecture.ipynb){download="chapter_08_lecture.ipynb"} &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/lshpaner/excel_to_python_exercises/blob/main/notebooks/chapter_08/chapter_08_lecture.ipynb){.colab-link}
:::

::: {.content-visible unless-format="html"}
Notebook and exercises for this chapter: github.com/lshpaner/excel_to_python_exercises, `notebooks/chapter_08/`.
:::

## Back to the decision

"Ridership is falling" is a claim about a slope. Chapter 7 compared January with March as two groups and could not separate them with 31 days a side. This chapter asks the same question a better way: instead of two piles of days, use all 90 in order and fit a line through them. A line has a slope, the slope has a standard error, and the pair is the direct answer to "is it falling, and how fast".

Then the chapter does what every regression does next: it looks at what the line got wrong, adds the column that explains the biggest pattern in the mistakes, and stops when the mistakes left over are things the data does not contain.

In [ ]:
try:
    import eda_toolkit
except ImportError:                      # Colab does not have it yet; local setups install it from requirements.txt
    %pip install -q eda_toolkit
    import eda_toolkit

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
import statsmodels.formula.api as smf
from eda_toolkit import scatter_fit_plot, flex_corr_matrix, plot_distributions

LOCAL = "../../data/raw/"
URL = "https://github.com/lshpaner/excel_to_python_exercises/releases/download/data/"
src = LOCAL if os.path.exists(LOCAL + "metro_trips.csv") else URL

trips = pd.read_csv(src + "metro_trips.csv")
trips["start_time"] = pd.to_datetime(trips["start_time"])
trips["date"] = trips["start_time"].dt.normalize()
trips = trips[(trips["duration"] > 1) & (trips["duration"] < 1440)].copy()   # chapter 3's keep mask

BLUE, ORANGE, LIGHT = "#1F4E79", "#C55A11", "#9DB4CC"
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
len(trips)

`statsmodels` is the new library. It fits regressions and reports them the way a statistics textbook does, with a standard error and a p-value per coefficient, which is what this chapter needs. Chapter 9 switches to `scikit-learn`, which fits the same models but reports them the way a prediction engineer wants.

## A table of days

A regression needs one row per unit and one column per thing you know about it. The unit here is a day, so the first step is the same `groupby` as chapter 5, with a few more columns: the day's number in the quarter (0 for January 1), its weekday, and three things about the day's trips that might travel with the count.

In [ ]:
day = (trips.groupby("date")
            .agg(trips=("trip_id", "size"),
                 mean_duration=("duration", "mean"),
                 electric_share=("bike_type", lambda s: (s == "electric").mean()),
                 walkup_share=("passholder_type", lambda s: (s == "Walk-up").mean()))
            .reset_index())
day["day_num"] = (day["date"] - day["date"].min()).dt.days
day["weekday"] = day["date"].dt.strftime("%a")
day["weekend"] = day["date"].dt.weekday >= 5
day.round(3).head()

Ninety rows. Everything in this chapter is fitted to this table, and the first lesson is that it took eight lines to build and will take one line to model. Building the table is the work.

### Correlation first

Before fitting anything, look at how the columns move together. A correlation is a number between $-1$ and $1$: $1$ means two columns rise and fall in lockstep, $-1$ means one rises as the other falls, $0$ means knowing one tells you nothing about the other. `flex_corr_matrix` draws every pair at once.

In [ ]:
#| label: fig-corr
#| fig-cap: "Correlations between the daily columns. Trips per day moves weakly with everything; the strong cells are among the other columns: Walk-up share and mean duration travel together (0.88), and electric share falls across the quarter (-0.67 with day of quarter)."
flex_corr_matrix(day[["trips", "mean_duration", "electric_share", "walkup_share", "day_num"]],
                 figsize=(6.5, 5), title="", show_plot=True, label_fontsize=11, tick_fontsize=9,
                 label_names={"trips": "trips per day", "mean_duration": "mean duration", "electric_share": "electric share",
                              "walkup_share": "Walk-up share", "day_num": "day of quarter"})

Before any model, the matrix already says two things. Trips per day correlates with day of quarter at 0.23: positive, weak, which is the "is it falling" answer in miniature and says the line will slope up, gently. And the strongest cell in the matrix is not about trips at all: days with more Walk-up riders are days with longer trips (0.88), which is chapter 7's weekend finding seen from a third angle. Excel's `CORREL(range1, range2)` gives one cell of this matrix; the Data Analysis ToolPak's Correlation tool gives the whole thing, without the colors.

## One line

The simplest regression is a straight line through a scatter: trips per day against day of the quarter. `scatter_fit_plot` draws it with the equation, which is what Excel's "Add trendline" with "Display equation on chart" does.

In [ ]:
#| label: fig-line
#| fig-cap: "Trips per day against day of the quarter, with the fitted line. The slope is 1.6 trips per day per day; the three low points near day 0 are January 1 to 4, and the one at day 46 is Presidents' Day."
scatter_fit_plot(day, x_vars=["day_num"], y_vars=["trips"], add_best_fit_line=True, show_plot="subplots",
                 scatter_color=BLUE, best_fit_linecolor=ORANGE, subplot_figsize=(9, 4),
                 label_names={"day_num": "Day of the quarter", "trips": "Trips"}, label_fontsize=11, tick_fontsize=9)

The line is $\text{trips} = 878 + 1.60 \times \text{day}$. The intercept, 878, is the line's value on day 0, January 1. The slope, 1.60, is the finding: each day of the quarter adds 1.6 trips to the daily count, which is 11 trips a week, or about 100 trips a day more by the end of March than the start of January. Excel gets both numbers with `SLOPE(trips_range, day_range)` and `INTERCEPT(trips_range, day_range)`.

What the chart cannot show is how sure the slope is. For that, fit the same line with `statsmodels` and read its table.

In [ ]:
line = smf.ols("trips ~ day_num", data=day).fit()
report = pd.DataFrame({"coefficient": line.params, "std error": line.bse, "p-value": line.pvalues,
                       "ci low": line.conf_int()[0], "ci high": line.conf_int()[1]}).round(3)
print(f"R-squared = {line.rsquared:.3f}")
report

The formula `"trips ~ day_num"` reads "trips as a function of day_num", and `ols` is ordinary least squares, the method that picks the line with the smallest total squared vertical distance to the points. Excel's `LINEST` does the same arithmetic.

Four columns to read, for the `day_num` row:

- **coefficient**, 1.60: the slope, as on the chart.
- **std error**, 0.72: chapter 7's standard error, now for a slope. Shuffle which 90 days you happened to get and the slope would wobble by about this much.
- **p-value**, 0.028: the chance of a slope this far from zero if the true slope were zero. Under the 0.05 line, so the upward drift is more than noise.
- **ci low** and **ci high**, 0.18 to 3.02: the 95% interval for the slope. The quarter is consistent with anything from a trickle (one more trip a week) to a surge (20 more a week). The interval, not the point, is the honest statement.

And one number above the table. **R-squared** is the share of the day-to-day variation in trips that the line accounts for, and 0.054 means 5%. The trend is real and it explains almost nothing about why one day has 900 trips and the next has 1,100. Both facts matter, and the next two sections chase the other 95%.

## Residuals: what the line gets wrong

A residual is actual minus fitted: how far above or below the line a day sits. Plot them in order and the pattern the line missed shows up.

In [ ]:
#| label: fig-resid
#| fig-cap: "Residuals from the straight line, one per day, weekends in orange. The weekend points sit above zero more often than below, which is a pattern the line did not know about; the deepest points are the first weekend of January and Presidents' Day."
day["resid_line"] = line.resid
fig, ax = plt.subplots()
colors = np.where(day["weekend"], ORANGE, BLUE)
ax.bar(day["date"], day["resid_line"], color=colors, width=0.8)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("Actual minus fitted (trips)")
ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
ax.set_title("Weekends (orange) sit above the line; holidays fall far below it", loc="left", fontsize=11)
plt.tight_layout()

The residuals are the chapter's method in one picture. Orange bars above zero on most weekends means the line is too low on Saturdays and Sundays and too high on weekdays, which is to say the weekday pattern from chapter 5 is sitting in the errors. A pattern in the residuals is a column the model is missing. Add it.

## Adding weekday

Weekday is a category, not a number, and a line cannot multiply "Tue" by a coefficient. The standard trick is to turn one category column into six yes-or-no columns, one per weekday except one, and give each its own coefficient. The left-out weekday is the reference, and every other coefficient is "compared with that day". `C(weekday, Treatment("Mon"))` tells `statsmodels` to do this with Monday as the reference; Excel needs the six 0/1 columns typed in by hand, `=IF(weekday="Tue", 1, 0)` and so on, before `LINEST` can see them.

In [ ]:
both = smf.ols('trips ~ day_num + C(weekday, Treatment("Mon"))', data=day).fit()
report = pd.DataFrame({"coefficient": both.params, "std error": both.bse, "p-value": both.pvalues}).round(2)
report.index = [i.replace('C(weekday, Treatment("Mon"))[T.', "").rstrip("]") for i in report.index]
print(f"R-squared = {both.rsquared:.3f}")
report

Read it as a sentence per row. The slope is still 1.6 trips a day, now "holding weekday fixed", which is what the extra columns buy: the trend is measured within Mondays, within Tuesdays, and so on, so a quarter that happened to have one more Saturday would not fake a trend. The weekday rows are each day's lift over a Monday on the same date: Saturdays run 171 trips above, Sundays 115, and Wednesdays, for whatever reason this quarter, 105. Tuesday's 38 has a p-value of 0.58, which means the data cannot tell a Tuesday from a Monday.

R-squared went from 0.054 to 0.139. Knowing the weekday more than doubled what the model can explain, and it still explains only 14% of the variation. Here are the two models over the actual counts.

In [ ]:
#| label: fig-fitted
#| fig-cap: "Actual trips per day (points) with the straight line and the line-plus-weekday model. The weekday model follows the weekly rhythm; neither model knows about the first week of January or Presidents' Day, because nothing in the table says 'holiday'."
day["fit_line"] = line.fittedvalues
day["fit_both"] = both.fittedvalues
fig, ax = plt.subplots()
ax.scatter(day["date"], day["trips"], color=BLUE, s=14, label="actual", zorder=3)
ax.plot(day["date"], day["fit_line"], color=LIGHT, lw=2, label="line only")
ax.plot(day["date"], day["fit_both"], color=ORANGE, lw=1.4, label="line + weekday")
ax.set_ylim(0, 1300); ax.set_ylabel("Trips per day")
ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
ax.legend(frameon=False, loc="lower right", ncol=3)
ax.set_title("The weekday model follows the week; neither model knows a holiday", loc="left", fontsize=11)
plt.tight_layout()

## What is left over

The model's five worst days:

In [ ]:
day["resid_both"] = both.resid
(day.sort_values("resid_both")[["date", "weekday", "trips", "fit_both", "resid_both"]]
    .head(5).round(0).set_index("date"))

January 2, 3 and 4, the first weekend of the year; February 16, Presidents' Day, with 224 clean trips (chapter 1 counted 238 before cleaning), 645 below what a Monday in mid-February should have had; and February 19, a Thursday with no holiday to blame, which is the shape of a rainy day. None of these is a mistake in the arithmetic. The table of days has a column for the calendar and no column for holidays or weather, and those are what move a day by 400 trips. The spread of the residuals tells the same story: the raw counts have a standard deviation of 180 trips; the straight line's residuals, 175; the weekday model's, 167. The calendar bought 13 trips a day of precision, and the rest is in columns the data does not have.

In [ ]:
#| label: fig-resid-hist
#| fig-cap: "Residuals from the line-plus-weekday model. Most days land within 150 trips of the model; the five in the left tail are the holidays and the one suspect Thursday."
plot_distributions(day, ["resid_both"], plot_type="hist", density_function=None, stat="count", binwidth=50,
                   hist_color=BLUE, hist_edgecolor="white", y_axis_label="Days", figsize=(8, 3.6),
                   plot_mean=False, plot_median=False, custom_titles={"resid_both": "Actual minus fitted, line + weekday model"},
                   custom_xlabels={"resid_both": "trips"}, label_fontsize=11, tick_fontsize=9)

This is where a regression on this table ends, and the ending is a finding in its own right: to explain Metro's daily counts you need a holiday flag and a weather feed, and a memo that asks for them is worth more than one that reports a slope to two decimals.

## Extrapolation

The line says trips rise 1.6 a day. Run it forward and it predicts about 1,320 trips a day at the start of October and 1,460 by late December.

In [ ]:
for label, d in [("first day of Q4", 273), ("last day of the year", 364)]:
    print(f"{label}: {line.params['Intercept'] + line.params['day_num'] * d:,.0f} trips")

Do not believe either number. The line was fitted to January, February and March, when days get longer and warmer, and it has no idea that the same thing runs backward in the fall. A regression describes the range it was fitted on, and every step outside that range is a guess dressed as a calculation. The slope is a finding about this quarter; the forecast is not a finding at all.

## Excel's regression, and where it stops

| Task | Excel | Python |
|---|---|---|
| One pair's correlation | `CORREL(x, y)` | `day[["x", "y"]].corr()` |
| Every pair at once | Data Analysis ToolPak, Correlation | `flex_corr_matrix(day)` |
| Slope and intercept | `SLOPE`, `INTERCEPT`; chart trendline | `smf.ols("y ~ x", day).fit()` |
| R-squared | `RSQ`; trendline option | `.rsquared` |
| Standard errors and p-values | `LINEST(y, x, TRUE, TRUE)`, five rows of unlabeled output; or the ToolPak's Regression | `.bse`, `.pvalues`, `.conf_int()` |
| A categorical predictor | six `IF` columns of 0/1, then `LINEST` | `C(weekday)` in the formula |
| Residuals | `y - (SLOPE*x + INTERCEPT)` per row | `.resid` |
| The fitted-vs-actual chart | two series on one chart | `.fittedvalues` and matplotlib |

Everything in the first four rows is one formula in Excel and the numbers agree exactly. The handoff is the fifth row onward. `LINEST` returns its standard errors as an unlabeled block that you have to know the layout of to read, the ToolPak's Regression dialog has to be rerun by hand every time the data changes, and a categorical predictor means building the dummy columns yourself and keeping them in sync with the data. In Python the formula names the columns, the fit returns a labeled table, and changing the model is editing one string.

## Back to the decision

Inside the quarter, ridership rose: 1.6 trips a day, 11 a week, with an interval of 1 to 21 a week, measured with weekdays held fixed. That is the opposite of the board's premise, and chapter 7 could not see it because it compared two months instead of fitting all 90 days. It is also a statement about three months, and the extrapolation section says why it cannot be stretched to a year.

The more useful finding is the residual one. The calendar explains 14% of day-to-day ridership and the remaining 86% is holidays and weather the data does not record. If Metro wants to know whether ridership is falling, it needs the same table of days with a weather column and four quarters in it. That request goes in the memo.

## Looking ahead

- **Worked exercise:** `chapter_08_guided.ipynb`. The same two models on Walk-up trips per day, where weekday explains 62% instead of 14%, and the reason why.
- **On your own:** `chapter_08_solo.ipynb`. The quarter's one strong trend: electric share per day falls from about 51% in the first week to 37% in the last. Fit it, read the table, check the residuals, and write the question Metro needs to answer before anyone acts on it.
- **Before the next chapter:** chapter 9 stops explaining and starts predicting. It asks whether a trip's pass type can be guessed from the trip itself, which is a different question from anything a slope answers. Think about which columns of a trip would give it away.

## Appendix: reproducing this chapter {.unnumbered}

`chapter_08_lecture.ipynb` is generated from this page; exercises come from `tools/build_chapter_08_exercises.py`. Models are fitted with `statsmodels.formula.api.ols` on a table of 90 days built from clean rows (the chapter 3 `keep` mask) in `data/raw/`; the correlation matrix, scatter with fit, and residual histogram come from `eda_toolkit`.